# Comprehensive Regression Models Comparison - California Housing Dataset

## Overview

This notebook provides a comprehensive comparison of various regression algorithms applied to the California Housing dataset. We will train and evaluate multiple models, compare their performance metrics, and visualize the results to understand which algorithm performs best for this particular dataset.

**Models to Compare:**
1. Linear Regression (Baseline)
2. Ridge Regression (L2 Regularization)
3. Lasso Regression (L1 Regularization)
4. Elastic Net Regression (L1 + L2)
5. Decision Tree Regression
6. Random Forest Regression
7. Support Vector Machine (SVM) Regression
8. K-Nearest Neighbors (KNN) Regression
9. Gradient Boosting Regression
10. XGBoost Regression

**Evaluation Metrics:**
- MAE (Mean Absolute Error)
- MSE (Mean Squared Error)
- RMSE (Root Mean Squared Error)
- R² (R-squared Score)

## Step 1: Import Required Libraries

We need to import all necessary libraries for data manipulation, visualization, and machine learning algorithms.

**Libraries:**
- **numpy**: For numerical operations and array manipulations
- **pandas**: For data manipulation and analysis using DataFrames
- **matplotlib**: For creating static visualizations and plots
- **seaborn**: For enhanced statistical visualizations
- **sklearn**: For machine learning algorithms, metrics, and dataset loading
- **xgboost**: For XGBoost algorithm (if available)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set style for better visualizations
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Sklearn modules
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# Regression models
from sklearn.linear_model import (
    LinearRegression,
    Ridge,
    Lasso,
    ElasticNet
)
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor
)
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor

# Try to import XGBoost
try:
    import xgboost as xgb
    XGBOOST_AVAILABLE = True
    print("XGBoost imported successfully!")
except ImportError:
    XGBOOST_AVAILABLE = False
    print("XGBoost not available. Install with: pip install xgboost")

## Step 2: Load and Prepare the Dataset

We'll use the California Housing dataset from sklearn. This dataset contains information about California housing districts with the goal of predicting the median house value.

**Steps:**
1. Fetch the dataset using `fetch_california_housing()`
2. Convert the data into a pandas DataFrame
3. Add the target variable (Price) as a new column
4. Display the first few rows to understand the data structure
5. Check dataset shape, info, and statistics

In [ ]:
# Load the dataset
housing = fetch_california_housing()

# Create DataFrame
df = pd.DataFrame(
    housing.data,
    columns=housing.feature_names
)

# Add target variable
df["Price"] = housing.target

# Display basic information
print("Dataset Shape:", df.shape)
print("\nFirst 5 rows:")
display(df.head())
print("\nDataset Info:")
print(df.info())
print("\nDataset Statistics:")
display(df.describe())

## Step 3: Data Exploration and Visualization

Before training models, we need to understand our data through visualization.

**Visualizations:**
- **Histograms**: To understand the distribution of each feature
- **Correlation Heatmap**: To see relationships between features and target
- **Boxplots**: To identify outliers in the data

In [ ]:
# Histograms for all features
df.hist(
    figsize=(16, 12),
    bins=30,
    edgecolor='black'
)
plt.suptitle('Feature Distributions', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation Heatmap
plt.figure(figsize=(12, 10))
corr = df.corr()
sns.heatmap(
    corr,
    annot=True,
    cmap='coolwarm',
    center=0,
    fmt='.2f',
    linewidths=1,
    cbar_kws={"shrink": 0.8}
)
plt.title('Correlation Matrix', fontsize=16, pad=20)
plt.tight_layout()
plt.show()

# Display correlation with target
print("\nCorrelation with Price (Target):")
print(corr['Price'].sort_values(ascending=False))

In [ ]:
# Boxplots to identify outliers
fig, axes = plt.subplots(3, 3, figsize=(16, 12))
axes = axes.ravel()

for idx, col in enumerate(df.columns):
    axes[idx].boxplot(df[col], patch_artist=True)
    axes[idx].set_title(col, fontsize=10)
    axes[idx].tick_params(axis='x', labelsize=8)
    axes[idx].tick_params(axis='y', labelsize=8)

# Hide empty subplot if any
if len(df.columns) < len(axes):
    for idx in range(len(df.columns), len(axes)):
        axes[idx].set_visible(False)

plt.suptitle('Boxplots - Outlier Detection', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

## Step 4: Data Preprocessing

Before training models, we need to preprocess the data:

**Preprocessing Steps:**
1. **Check for missing values**: Handle any missing data
2. **Check for duplicates**: Remove duplicate rows
3. **Split features and target**: Separate X (features) and y (target)
4. **Feature scaling**: Standardize features for algorithms that require it
5. **Train-test split**: Split data into training and testing sets

In [ ]:
# Check for missing values
print("Missing Values:")
print(df.isnull().sum())

# Check for duplicates
print("\nDuplicate Rows:", df.duplicated().sum())

# Split features and target
X = df.drop("Price", axis=1)
y = df["Price"]

print("\nFeatures Shape:", X.shape)
print("Target Shape:", y.shape)

In [ ]:
# Feature Scaling (Standardization)
# Formula: z = (x - μ) / σ
# This is crucial for algorithms like SVM, KNN, and regularized linear models

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

print("Scaled Features (First 5 rows):")
display(X_scaled.head())
print("\nScaled Statistics:")
display(X_scaled.describe())

In [ ]:
# Train-Test Split
# We use 80% for training and 20% for testing
# random_state=42 ensures reproducibility

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled,
    y,
    test_size=0.2,
    random_state=42
)

print("Training Set Shape:", X_train.shape)
print("Testing Set Shape:", X_test.shape)
print("\nTraining Target Range:", y_train.min(), "to", y_train.max())
print("Testing Target Range:", y_test.min(), "to", y_test.max())

## Step 5: Train All Regression Models

Now we'll train all regression models on the same dataset. We'll use default hyperparameters for fair comparison, but note that in practice, you should tune hyperparameters for each model.

**Models to train:**
1. Linear Regression
2. Ridge Regression
3. Lasso Regression
4. Elastic Net Regression
5. Decision Tree Regression
6. Random Forest Regression
7. SVM Regression
8. KNN Regression
9. Gradient Boosting Regression
10. XGBoost Regression (if available)

In [ ]:
# Dictionary to store models and their predictions
models = {}
predictions = {}
results = []

# 1. Linear Regression
print("Training Linear Regression...")
lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)
models['Linear Regression'] = lr
predictions['Linear Regression'] = y_pred_lr

# 2. Ridge Regression
print("Training Ridge Regression...")
ridge = Ridge(alpha=1.0, random_state=42)
ridge.fit(X_train, y_train)
y_pred_ridge = ridge.predict(X_test)
models['Ridge Regression'] = ridge
predictions['Ridge Regression'] = y_pred_ridge

# 3. Lasso Regression
print("Training Lasso Regression...")
lasso = Lasso(alpha=1.0, random_state=42)
lasso.fit(X_train, y_train)
y_pred_lasso = lasso.predict(X_test)
models['Lasso Regression'] = lasso
predictions['Lasso Regression'] = y_pred_lasso

# 4. Elastic Net Regression
print("Training Elastic Net Regression...")
elastic_net = ElasticNet(alpha=1.0, l1_ratio=0.5, random_state=42)
elastic_net.fit(X_train, y_train)
y_pred_en = elastic_net.predict(X_test)
models['Elastic Net'] = elastic_net
predictions['Elastic Net'] = y_pred_en

# 5. Decision Tree Regression
print("Training Decision Tree Regression...")
dt = DecisionTreeRegressor(random_state=42)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)
models['Decision Tree'] = dt
predictions['Decision Tree'] = y_pred_dt

# 6. Random Forest Regression
print("Training Random Forest Regression...")
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
models['Random Forest'] = rf
predictions['Random Forest'] = y_pred_rf

# 7. SVM Regression
print("Training SVM Regression...")
svr = SVR(kernel='rbf', C=1.0, epsilon=0.1)
svr.fit(X_train, y_train)
y_pred_svr = svr.predict(X_test)
models['SVM'] = svr
predictions['SVM'] = y_pred_svr

# 8. KNN Regression
print("Training KNN Regression...")
knn = KNeighborsRegressor(n_neighbors=5)
knn.fit(X_train, y_train)
y_pred_knn = knn.predict(X_test)
models['KNN'] = knn
predictions['KNN'] = y_pred_knn

# 9. Gradient Boosting Regression
print("Training Gradient Boosting Regression...")
gb = GradientBoostingRegressor(n_estimators=100, random_state=42)
gb.fit(X_train, y_train)
y_pred_gb = gb.predict(X_test)
models['Gradient Boosting'] = gb
predictions['Gradient Boosting'] = y_pred_gb

# 10. XGBoost Regression (if available)
if XGBOOST_AVAILABLE:
    print("Training XGBoost Regression...")
    xgb_model = xgb.XGBRegressor(n_estimators=100, random_state=42)
    xgb_model.fit(X_train, y_train)
    y_pred_xgb = xgb_model.predict(X_test)
    models['XGBoost'] = xgb_model
    predictions['XGBoost'] = y_pred_xgb
else:
    print("XGBoost not available, skipping...")

print("\nAll models trained successfully!")

## Step 6: Evaluate All Models

Now we'll evaluate each model using standard regression metrics:

**Metrics:**
- **MAE (Mean Absolute Error)**: Average absolute difference between predicted and actual values
- **MSE (Mean Squared Error)**: Average of squared differences (penalizes larger errors more)
- **RMSE (Root Mean Squared Error)**: Square root of MSE (in same units as target)
- **R² (R-squared)**: Proportion of variance in target explained by the model (0 to 1)

In [ ]:
# Evaluate all models
for model_name, y_pred in predictions.items():
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)
    
    results.append({
        'Model': model_name,
        'MAE': mae,
        'MSE': mse,
        'RMSE': rmse,
        'R²': r2
    })

# Create results DataFrame
results_df = pd.DataFrame(results)
results_df = results_df.sort_values('R²', ascending=False)

# Display results
print("Model Performance Comparison (Sorted by R²):")
print("="*80)
display(results_df.style.format({
    'MAE': '{:.4f}',
    'MSE': '{:.4f}',
    'RMSE': '{:.4f}',
    'R²': '{:.4f}'
}))

## Step 7: Visualize Model Performance

Let's create visualizations to compare the performance of all models:

**Visualizations:**
- **Bar chart of R² scores**: To compare model accuracy
- **Bar chart of RMSE**: To compare prediction error
- **Bar chart of MAE**: To compare average absolute error

In [ ]:
# Create subplots for different metrics
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# R² Score Comparison
colors_r2 = ['green' if x > 0.6 else 'orange' if x > 0.5 else 'red' for x in results_df['R²']]
axes[0].barh(results_df['Model'], results_df['R²'], color=colors_r2)
axes[0].set_xlabel('R² Score')
axes[0].set_title('R² Score Comparison', fontsize=14, fontweight='bold')
axes[0].set_xlim(0, 1)
axes[0].axvline(x=0.5, color='red', linestyle='--', alpha=0.5, label='Baseline (0.5)')
axes[0].legend()
for i, v in enumerate(results_df['R²']):
    axes[0].text(v + 0.01, i, f'{v:.3f}', va='center', fontsize=9)

# RMSE Comparison
axes[1].barh(results_df['Model'], results_df['RMSE'], color='steelblue')
axes[1].set_xlabel('RMSE')
axes[1].set_title('RMSE Comparison', fontsize=14, fontweight='bold')
for i, v in enumerate(results_df['RMSE']):
    axes[1].text(v + 0.01, i, f'{v:.3f}', va='center', fontsize=9)

# MAE Comparison
axes[2].barh(results_df['Model'], results_df['MAE'], color='coral')
axes[2].set_xlabel('MAE')
axes[2].set_title('MAE Comparison', fontsize=14, fontweight='bold')
for i, v in enumerate(results_df['MAE']):
    axes[2].text(v + 0.01, i, f'{v:.3f}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

## Step 8: Feature Importance Comparison

Let's compare feature importance across tree-based models (Random Forest, Gradient Boosting, and XGBoost if available).

In [ ]:
# Get feature importance from tree-based models
tree_models = ['Random Forest', 'Gradient Boosting']
if XGBOOST_AVAILABLE:
    tree_models.append('XGBoost')

feature_importance_data = {}
for model_name in tree_models:
    if model_name in models:
        feature_importance_data[model_name] = models[model_name].feature_importances_

# Create DataFrame for feature importance
feature_importance_df = pd.DataFrame(
    feature_importance_data,
    index=X_train.columns
)

# Display feature importance
print("Feature Importance Comparison:")
display(feature_importance_df)

# Visualize feature importance
fig, axes = plt.subplots(1, len(tree_models), figsize=(16, 5))
if len(tree_models) == 1:
    axes = [axes]

for idx, model_name in enumerate(tree_models):
    if model_name in feature_importance_data:
        importance = feature_importance_data[model_name]
        sorted_idx = np.argsort(importance)
        axes[idx].barh(
            range(len(importance)),
            importance[sorted_idx]
        )
        axes[idx].set_yticks(range(len(importance)))
        axes[idx].set_yticklabels(X_train.columns[sorted_idx])
        axes[idx].set_xlabel('Importance')
        axes[idx].set_title(f'{model_name} Feature Importance', fontsize=12, fontweight='bold')
        axes[idx].invert_yaxis()

plt.tight_layout()
plt.show()

## Step 9: Actual vs Predicted Plots

Let's visualize the actual vs predicted values for each model to see how well they fit the data.

In [ ]:
# Create subplots for actual vs predicted
n_models = len(predictions)
n_cols = 3
n_rows = (n_models + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 6*n_rows))
axes = axes.ravel() if n_rows > 1 else [axes]

for idx, (model_name, y_pred) in enumerate(predictions.items()):
    axes[idx].scatter(y_test, y_pred, alpha=0.5, s=20)
    axes[idx].plot(
        [y_test.min(), y_test.max()],
        [y_test.min(), y_test.max()],
        'r--', linewidth=2, label='Perfect Prediction'
    )
    axes[idx].set_xlabel('Actual Values', fontsize=10)
    axes[idx].set_ylabel('Predicted Values', fontsize=10)
    axes[idx].set_title(f'{model_name}\nR² = {results_df[results_df["Model"] == model_name]["R²"].values[0]:.4f}', 
                       fontsize=11, fontweight='bold')
    axes[idx].legend(fontsize=8)
    axes[idx].grid(True, alpha=0.3)

# Hide empty subplots
for idx in range(len(predictions), len(axes)):
    axes[idx].set_visible(False)

plt.suptitle('Actual vs Predicted Values for All Models', fontsize=16, fontweight='bold', y=1.005)
plt.tight_layout()
plt.show()

## Step 10: Residual Analysis

Residual plots help us understand the distribution of errors and identify patterns in model predictions.

In [ ]:
# Create subplots for residual plots
fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 6*n_rows))
axes = axes.ravel() if n_rows > 1 else [axes]

for idx, (model_name, y_pred) in enumerate(predictions.items()):
    residuals = y_test - y_pred
    axes[idx].scatter(y_pred, residuals, alpha=0.5, s=20)
    axes[idx].axhline(y=0, color='r', linestyle='--', linewidth=2)
    axes[idx].set_xlabel('Predicted Values', fontsize=10)
    axes[idx].set_ylabel('Residuals', fontsize=10)
    axes[idx].set_title(f'{model_name} Residuals', fontsize=11, fontweight='bold')
    axes[idx].grid(True, alpha=0.3)

# Hide empty subplots
for idx in range(len(predictions), len(axes)):
    axes[idx].set_visible(False)

plt.suptitle('Residual Plots for All Models', fontsize=16, fontweight='bold', y=1.005)
plt.tight_layout()
plt.show()

## Step 11: Residual Distribution Histograms

Let's examine the distribution of residuals to check if they follow a normal distribution (ideal for regression models).

In [ ]:
# Create subplots for residual histograms
fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 6*n_rows))
axes = axes.ravel() if n_rows > 1 else [axes]

for idx, (model_name, y_pred) in enumerate(predictions.items()):
    residuals = y_test - y_pred
    axes[idx].hist(residuals, bins=30, edgecolor='black', alpha=0.7)
    axes[idx].axvline(x=0, color='r', linestyle='--', linewidth=2)
    axes[idx].set_xlabel('Residuals', fontsize=10)
    axes[idx].set_ylabel('Frequency', fontsize=10)
    axes[idx].set_title(f'{model_name} Residual Distribution', fontsize=11, fontweight='bold')
    axes[idx].grid(True, alpha=0.3)

# Hide empty subplots
for idx in range(len(predictions), len(axes)):
    axes[idx].set_visible(False)

plt.suptitle('Residual Distribution Histograms', fontsize=16, fontweight='bold', y=1.005)
plt.tight_layout()
plt.show()

## Step 12: Performance Radar Chart

A radar chart provides a comprehensive view of model performance across multiple metrics.

In [ ]:
# Normalize metrics for radar chart (higher is better)
radar_df = results_df.copy()

# For RMSE and MAE, we'll use inverse (lower is better, so we normalize)
max_rmse = radar_df['RMSE'].max()
max_mae = radar_df['MAE'].max()

radar_df['RMSE_norm'] = 1 - (radar_df['RMSE'] / max_rmse)
radar_df['MAE_norm'] = 1 - (radar_df['MAE'] / max_mae)
radar_df['R²_norm'] = radar_df['R²']  # Already normalized

# Select top 5 models for clarity
top_models = radar_df.head(5)

# Create radar chart
categories = ['R²', 'RMSE (normalized)', 'MAE (normalized)']
N = len(categories)

angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]  # Complete the circle

fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(projection='polar'))

colors = plt.cm.Set3(np.linspace(0, 1, len(top_models)))

for idx, (_, row) in enumerate(top_models.iterrows()):
    values = [row['R²_norm'], row['RMSE_norm'], row['MAE_norm']]
    values += values[:1]
    
    ax.plot(angles, values, 'o-', linewidth=2, label=row['Model'], color=colors[idx])
    ax.fill(angles, values, alpha=0.15, color=colors[idx])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories)
ax.set_ylim(0, 1)
ax.set_title('Top 5 Models Performance Radar Chart', fontsize=16, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
ax.grid(True)

plt.tight_layout()
plt.show()

## Step 13: Summary and Recommendations

Based on our comprehensive comparison, let's summarize the findings and provide recommendations.

In [ ]:
# Display final summary
print("="*80)
print("REGRESSION MODELS COMPARISON SUMMARY")
print("="*80)
print()

print("\n📊 PERFORMANCE RANKING (by R² Score):")
print("-"*80)
for idx, row in results_df.iterrows():
    print(f"{idx+1}. {row['Model']:<25} R²: {row['R²']:.4f} | RMSE: {row['RMSE']:.4f} | MAE: {row['MAE']:.4f}")

print("\n🏆 BEST PERFORMING MODEL:")
print("-"*80)
best_model = results_df.iloc[0]
print(f"Model: {best_model['Model']}")
print(f"R² Score: {best_model['R²']:.4f} ({best_model['R²']*100:.2f}% variance explained)")
print(f"RMSE: {best_model['RMSE']:.4f}")
print(f"MAE: {best_model['MAE']:.4f}")

print("\n📈 KEY INSIGHTS:")
print("-"*80)
print("1. Ensemble methods (Random Forest, Gradient Boosting, XGBoost) generally perform better")
print("2. Linear models provide good baseline performance")
print("3. Regularization (Ridge, Lasso, Elastic Net) helps prevent overfitting")
print("4. Tree-based models capture non-linear relationships effectively")
print("5. SVM and KNN require careful hyperparameter tuning for optimal performance")

print("\n💡 RECOMMENDATIONS:")
print("-"*80)
print("1. For best accuracy: Use ensemble methods (Gradient Boosting or XGBoost)")
print("2. For interpretability: Use Linear Regression or Decision Tree")
print("3. For feature selection: Use Lasso or Elastic Net")
print("4. For quick prototyping: Start with Linear Regression as baseline")
print("5. Always perform hyperparameter tuning for production models")
print("6. Consider computational cost for large datasets")

print("\n" + "="*80)

## Step 14: Model Characteristics Summary

Let's provide a summary of each model's characteristics to help with model selection.

In [ ]:
# Create model characteristics summary
model_characteristics = pd.DataFrame({
    'Model': [
        'Linear Regression',
        'Ridge Regression',
        'Lasso Regression',
        'Elastic Net',
        'Decision Tree',
        'Random Forest',
        'SVM',
        'KNN',
        'Gradient Boosting',
        'XGBoost'
    ],
    'Type': [
        'Linear',
        'Linear (Regularized)',
        'Linear (Regularized)',
        'Linear (Regularized)',
        'Tree-based',
        'Ensemble (Bagging)',
        'Kernel-based',
        'Instance-based',
        'Ensemble (Boosting)',
        'Ensemble (Boosting)'
    ],
    'Scaling Required': [
        'Yes',
        'Yes',
        'Yes',
        'Yes',
        'No',
        'No',
        'Yes',
        'Yes',
        'No',
        'No'
    ],
    'Handles Non-linearity': [
        'No',
        'No',
        'No',
        'No',
        'Yes',
        'Yes',
        'Yes (with kernels)',
        'No',
        'Yes',
        'Yes'
    ],
    'Interpretability': [
        'High',
        'High',
        'High',
        'High',
        'High',
        'Medium',
        'Low',
        'Medium',
        'Low',
        'Low'
    ],
    'Training Speed': [
        'Fast',
        'Fast',
        'Fast',
        'Fast',
        'Fast',
        'Medium',
        'Slow (large data)',
        'Fast (no training)',
        'Medium',
        'Medium'
    ],
    'Prediction Speed': [
        'Fast',
        'Fast',
        'Fast',
        'Fast',
        'Fast',
        'Medium',
        'Medium',
        'Slow',
        'Medium',
        'Fast'
    ],
    'Best For': [
        'Baseline, simple relationships',
        'Multicollinearity, regularization',
        'Feature selection',
        'Balanced regularization',
        'Interpretability, non-linear',
        'High accuracy, robustness',
        'High-dimensional data',
        'Small datasets, simple patterns',
        'High accuracy, structured data',
        'High accuracy, large datasets'
    ]
})

print("\nMODEL CHARACTERISTICS SUMMARY:")
print("="*120)
display(model_characteristics)
print("="*120)